## Imports

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    upper,
    trim,
    to_date,
    current_timestamp,
    row_number,
    lit,
    when,
    lpad
)
from pyspark.sql.window import Window

## Bronze table

In [0]:
bronze_df = spark.table("bronze_payments")
silver_customers = spark.table("silver_customers")
silver_orders = spark.table("silver_orders")

display(bronze_df.limit(10))
print(f"Bronze rows: {bronze_df.count():,}")

payment_id,order_id,customer_id,payment_date,payment_status,payment_method,payment_amount,currency,created_at,updated_at,source_system
PAY000000001,ORD000000001,CRM0039575,2023-07-17T16:35:05.996Z,SUCCESS,GOOGLE_PAY,1133.55,AUD,2023-07-17T16:35:05.996Z,2023-07-17T16:35:05.996Z,payment_gateway
PAY000000002,ORD000000002,CRM0039575,2025-07-31T19:03:05.996Z,SUCCESS,CARD,3470.62,AUD,2025-07-31T19:03:05.996Z,2025-08-09T19:03:05.996Z,payment_gateway
PAY000000003,ORD000000003,CRM0039575,2024-08-18T18:06:05.996Z,SUCCESS,CARD,3089.93,AUD,2024-08-18T18:06:05.996Z,2024-08-27T18:06:05.996Z,payment_gateway
PAY000000004,ORD000000004,CRM0039575,2024-09-09T17:21:05.996Z,SUCCESS,APPLE_PAY,1455.64,AUD,2024-09-09T17:21:05.996Z,2024-09-14T17:21:05.996Z,payment_gateway
PAY000000005,ORD000000005,CRM0039575,2025-05-22T17:56:05.996Z,SUCCESS,BANK_TRANSFER,3885.04,AUD,2025-05-22T17:56:05.996Z,2025-06-01T17:56:05.996Z,payment_gateway
PAY000000006,ORD000000006,CRM0039575,2022-06-24T17:22:05.996Z,SUCCESS,BANK_TRANSFER,4223.83,AUD,2022-06-24T17:22:05.996Z,2022-07-04T17:22:05.996Z,payment_gateway
PAY000000007,ORD000000007,CRM0039575,2023-11-30T18:57:05.996Z,PENDING,CARD,2422.42,AUD,2023-11-30T18:57:05.996Z,2023-12-03T18:57:05.996Z,payment_gateway
PAY000000008,ORD000000008,CRM0000538,2021-10-01T17:54:05.996Z,SUCCESS,PAYPAL,1123.5,AUD,2021-10-01T17:54:05.996Z,2021-10-10T17:54:05.996Z,payment_gateway
PAY000000009,ORD000000009,CRM0000538,2023-05-31T18:53:05.996Z,SUCCESS,CARD,2028.61,AUD,2023-05-31T18:53:05.996Z,2023-06-09T18:53:05.996Z,payment_gateway
PAY000000010,ORD000000010,CRM0000538,2022-06-10T17:34:05.996Z,SUCCESS,PAYPAL,3382.58,AUD,2022-06-10T17:34:05.996Z,2022-06-12T17:34:05.996Z,payment_gateway


Bronze rows: 446,598


## Silver transformation

In [0]:

PAYMENT_STATUSES = [
    "SUCCESS",
    "FAILED",
    "PENDING",
    "REFUNDED"
]
PAYMENT_METHODS = [
    "CARD",
    "PAYPAL",
    "APPLE_PAY",
    "GOOGLE_PAY",
    "BANK_TRANSFER"
]

standardised_df = (
    bronze_df
    .withColumn("payment_date", to_date(col("payment_date")))
    .withColumn("payment_status", upper(trim(col("payment_status"))))
    .withColumn("payment_method", upper(trim(col("payment_method"))))
    .withColumn("payment_amount", col("payment_amount"))
    .withColumn("currency", upper(trim(col("currency"))))
    .withColumn("created_at", to_date(col("created_at")))
    .withColumn("updated_at", to_date(col("updated_at")))
    .withColumn("_silver_processed_at", current_timestamp())
) 
window_spec = Window.partitionBy("payment_id").orderBy(col("updated_at").desc_nulls_last()) 
deduped_df = (
    standardised_df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") ==1)
    .drop("row_num")
)
customer_lookup = (
    silver_customers
    .select("customer_id")
    .dropDuplicates(["customer_id"])
    .withColumn("customer_exists", lit(True))
) 
order_lookup = (
    silver_orders
    .select("order_id")
    .dropDuplicates(["order_id"])
    .withColumn("order_exists", lit(True))
) 
payments_with_customer_order_check = (
    deduped_df
    .join(
        customer_lookup,
        on="customer_id",
        how= "left"
    )
    .join(
        order_lookup,
        on="order_id",
        how = "left"
    )
)
quality_df = (
    payments_with_customer_order_check
    .withColumn(
        "_has_future_updated_at",
        col("updated_at") > current_timestamp()
    )
    .withColumn(
        "_is_valid",
        when(col("payment_id").isNull(), lit(False))
        .when(col("order_id").isNull(), lit(False))
        .when(col("order_exists").isNull(), lit(False))
        .when(col("customer_id").isNull(), lit(False))
        .when(col("customer_exists").isNull(), lit(False))
        .when(col("payment_date").isNull(), lit(False))
        .when(col("payment_status").isNull(), lit(False))
        .when(~col("payment_status").isin(PAYMENT_STATUSES), lit(False))
        .when(col("payment_method").isNull(), lit(False))
        .when(~col("payment_method").isin(PAYMENT_METHODS), lit(False))
        .when(col("payment_amount") <= 0, lit(False))
        .otherwise(lit(True))
    )
    .withColumn(
        "_rejection_reason",
        when(col("payment_id").isNull(), lit("payment_id is null"))
        .when(col("order_id").isNull(), lit("order_id is null"))
        .when(col("order_exists").isNull(), lit("order_id does not exist in silver orders"))
        .when(col("customer_id").isNull(), lit("customer_id is null"))
        .when(col("customer_exists").isNull(), lit("customer_id does not exist in silver customers"))
        .when(col("payment_date").isNull(), lit("payment_date is null"))
        .when(col("payment_status").isNull(), lit("payment_status is null"))
        .when(~col("payment_status").isin(PAYMENT_STATUSES), lit("invalid payment_status"))
        .when(col("payment_method").isNull(), lit("payment_method is null"))
        .when(~col("payment_method").isin(PAYMENT_METHODS), lit("invalid payment_method"))
        .when(col("payment_amount") <= 0, lit("payment_amount must be greater than zero"))
        .otherwise(lit(None))
    )
) 
valid_payments = quality_df.filter(col("_is_valid")== True)
rejected_payments = quality_df.filter(col("_is_valid") == False)

## Delta tables

In [0]:
(
    valid_payments
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_payments")
)

(
    rejected_payments
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("rejected_payments")
)

print("Silver Payments completed")
print(f"Bronze rows: {bronze_df.count():,}")
print(f"After deduplication: {deduped_df.count():,}")
print(f"Valid payments: {valid_payments.count():,}")
print(f"Rejected payments: {rejected_payments.count():,}")

Silver Payments completed
Bronze rows: 446,598
After deduplication: 443,054
Valid payments: 430,303
Rejected payments: 12,751


In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|  bronze_addresses|      false|
| default|  bronze_customers|      false|
| default|    bronze_loyalty|      false|
| default|     bronze_orders|      false|
| default|   bronze_payments|      false|
| default|rejected_addresses|      false|
| default|rejected_customers|      false|
| default|   rejected_orders|      false|
| default| rejected_payments|      false|
| default|  silver_addresses|      false|
| default|  silver_customers|      false|
| default|     silver_orders|      false|
| default|   silver_payments|      false|
+--------+------------------+-----------+

